In [ ]:
import pandas as pd
import re

In [ ]:
nri_2023 = pd.read_csv("../../01_original_data/NRI/2023/NRI_Table_Counties.csv")
nri_2021 = pd.read_csv("../../01_original_data/NRI/2021/NRI_Table_Counties.csv")
nri_2020 = pd.read_csv("../../01_original_data/NRI/2020/NRI_Table_Counties.csv")

nri = pd.concat([nri_2023, nri_2021, nri_2020], ignore_index=True)
nri.columns = [re.sub(r"\s+", "_", c.strip().lower()) for c in nri.columns]
nri["stcofips"] = nri["statefips"].astype(str).str.zfill(2) + nri["countyfips"].astype(
    str
).str.zfill(3)

# Comment for full NRI
# nri = nri[
#     [
#         "state",
#         "stateabbrv",
#         "county",
#         "statefips",
#         "stcofips",
#         "population",
#         "risk_value",
#         "risk_score",
#         "risk_ratng",
#         "nri_ver",
#     ]
# ]

nri["nri_ver"] = (
    nri["nri_ver"]
    .map({"October 2020": 2020, "November 2021": 2021, "March 2023": 2023})
    .astype(int)
)

# nri.info()
print(len(nri))
nri.head(10)

9515


,oid_,nri_id,state,stateabbrv,statefips,county,countytype,countyfips,stcofips,population,...,wntw_alr_npctl,wntw_riskv,wntw_risks,wntw_riskr,nri_ver,risk_npctl,eal_npctl,sovi_npctl,sovi_value,resl_npctl
0,1,C01001,Alabama,AL,1,Autauga,County,1,01001,58764,...,10.461158,8494.906508,12.217626,Very Low,2023,NaN,NaN,NaN,NaN,NaN
1,2,C01003,Alabama,AL,1,Baldwin,County,3,01003,231365,...,13.339523,65619.701638,52.083996,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN
2,3,C01005,Alabama,AL,1,Barbour,County,5,01005,25160,...,16.125039,15501.730335,19.535476,Very Low,2023,NaN,NaN,NaN,NaN,NaN
3,4,C01007,Alabama,AL,1,Bibb,County,7,01007,22239,...,16.991643,7496.186940,11.104041,Very Low,2023,NaN,NaN,NaN,NaN,NaN
4,5,C01009,Alabama,AL,1,Blount,County,9,01009,58992,...,12.039616,17175.160729,21.444480,Very Low,2023,NaN,NaN,NaN,NaN,NaN
5,6,C01011,Alabama,AL,1,Bullock,County,11,01011,10326,...,20.458063,7340.314706,10.881324,Very Low,2023,NaN,NaN,NaN,NaN,NaN
6,7,C01013,Alabama,AL,1,Butler,County,13,01013,19015,...,17.239245,9478.883530,13.299395,Very Low,2023,NaN,NaN,NaN,NaN,NaN
7,8,C01015,Alabama,AL,1,Calhoun,County,15,01015,116250,...,11.451563,35926.190293,37.671015,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN
8,9,C01017,Alabama,AL,1,Chambers,County,17,01017,34738,...,11.761065,8875.448047,12.663061,Very Low,2023,NaN,NaN,NaN,NaN,NaN
9,10,C01019,Alabama,AL,1,Cherokee,County,19,01019,24933,...,15.475085,8944.754755,12.726694,Very Low,2023,NaN,NaN,NaN,NaN,NaN


In [162]:
hpi = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/hpi_master.csv"
)
# print(min(hpi["yr"]))
hpi = hpi[
    (hpi["level"] == "MSA") & (hpi["yr"].isin([2019, 2020, 2021, 2022, 2023]))
]  # only keep data for msa level
hpi = hpi[["place_name", "place_id", "yr", "period", "index_nsa"]]
# hpi.info()
# hpi.head(10)

In [163]:
hpi = hpi.groupby(["place_id", "yr"], as_index=False).agg(
    {
        "place_name": "first",
        "place_id": "first",
        "yr": "first",
        "index_nsa": "mean",
    }
)

hpi["index_prev"] = hpi.groupby(["place_id"])["index_nsa"].shift(1)
hpi = hpi[hpi["yr"].isin([2020, 2021, 2022, 2023])]
hpi["hpi_change"] = hpi["index_nsa"] - hpi["index_prev"]
hpi

,place_name,place_id,yr,index_nsa,index_prev,hpi_change
1,"Abilene, TX",10180,2020,235.9400,225.7875,10.1525
2,"Abilene, TX",10180,2021,266.3675,235.9400,30.4275
3,"Abilene, TX",10180,2022,300.3175,266.3675,33.9500
4,"Abilene, TX",10180,2023,327.3250,300.3175,27.0075
6,"Akron, OH",10420,2020,170.0325,160.8900,9.1425
...,...,...,...,...,...,...
2044,"Yuba City, CA",49700,2023,359.3725,362.1900,-2.8175
2046,"Yuma, AZ",49740,2020,207.5775,193.8850,13.6925
2047,"Yuma, AZ",49740,2021,249.3650,207.5775,41.7875
2048,"Yuma, AZ",49740,2022,302.8575,249.3650,53.4925


In [164]:
xwalk = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/list1_2023.csv"
)
xwalk.columns = (
    xwalk.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
)
xwalk["county_fips5"] = (
    xwalk["fips_state_code"].astype(str).str.upper().str.strip().str.zfill(2)
    + xwalk["fips_county_code"].astype(str).str.zfill(3).str.strip()
)
xwalk = xwalk[
    [
        "cbsa_code",
        "cbsa_title",
        "county_fips5",
        "county/county_equivalent",
        "state_name",
    ]
]
xwalk["cbsa_code"] = xwalk["cbsa_code"].astype(str)
# xwalk.info()
print(len(xwalk))
xwalk.head(10)

1915


,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name
0,10100,"Aberdeen, SD",46013,Brown County,South Dakota
1,10100,"Aberdeen, SD",46045,Edmunds County,South Dakota
2,10140,"Aberdeen, WA",53027,Grays Harbor County,Washington
3,10180,"Abilene, TX",48059,Callahan County,Texas
4,10180,"Abilene, TX",48253,Jones County,Texas
5,10180,"Abilene, TX",48441,Taylor County,Texas
6,10220,"Ada, OK",40123,Pontotoc County,Oklahoma
7,10300,"Adrian, MI",26091,Lenawee County,Michigan
8,10380,"Aguadilla, PR",72003,Aguada Municipio,Puerto Rico
9,10380,"Aguadilla, PR",72005,Aguadilla Municipio,Puerto Rico


In [165]:
# merge NRI with crosswalk on county fips
nri = nri.merge(
    xwalk[["cbsa_code", "county_fips5"]],
    left_on="stcofips",
    right_on="county_fips5",
    how="left",
).dropna(subset="cbsa_code")
print(len(nri))
nri.head(10)

5576


,oid_,nri_id,state,stateabbrv,statefips,county,countytype,countyfips,stcofips,population,...,wntw_risks,wntw_riskr,nri_ver,risk_npctl,eal_npctl,sovi_npctl,sovi_value,resl_npctl,cbsa_code,county_fips5
0,1,C01001,Alabama,AL,1,Autauga,County,1,01001,58764,...,12.217626,Very Low,2023,NaN,NaN,NaN,NaN,NaN,33860,01001
1,2,C01003,Alabama,AL,1,Baldwin,County,3,01003,231365,...,52.083996,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN,19300,01003
2,3,C01005,Alabama,AL,1,Barbour,County,5,01005,25160,...,19.535476,Very Low,2023,NaN,NaN,NaN,NaN,NaN,21640,01005
3,4,C01007,Alabama,AL,1,Bibb,County,7,01007,22239,...,11.104041,Very Low,2023,NaN,NaN,NaN,NaN,NaN,13820,01007
4,5,C01009,Alabama,AL,1,Blount,County,9,01009,58992,...,21.444480,Very Low,2023,NaN,NaN,NaN,NaN,NaN,13820,01009
7,8,C01015,Alabama,AL,1,Calhoun,County,15,01015,116250,...,37.671015,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN,11500,01015
8,9,C01017,Alabama,AL,1,Chambers,County,17,01017,34738,...,12.663061,Very Low,2023,NaN,NaN,NaN,NaN,NaN,29300,01017
10,11,C01021,Alabama,AL,1,Chilton,County,21,01021,44999,...,16.576519,Very Low,2023,NaN,NaN,NaN,NaN,NaN,13820,01021
15,16,C01031,Alabama,AL,1,Coffee,County,31,01031,53391,...,48.075087,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN,21460,01031
16,17,C01033,Alabama,AL,1,Colbert,County,33,01033,57133,...,33.566656,Relatively Low,2023,NaN,NaN,NaN,NaN,NaN,22520,01033


In [166]:
# merge NRI with HPI on msa code
# print(hpi["yr"].value_counts())

nri = nri.merge(
    hpi,
    left_on=["cbsa_code", "nri_ver"],
    right_on=["place_id", "yr"],
    how="left",
).dropna(subset="index_nsa")

nri["yr"] = nri["yr"].astype(int)

print(nri.isna().sum())
print(len(nri))
nri.head(10)
# nri["yr"].value_counts()
# nri[nri["yr"].isna()]

oid_          0
nri_id        0
state         0
stateabbrv    0
statefips     0
             ..
place_id      0
yr            0
index_nsa     0
index_prev    0
hpi_change    0
Length: 478, dtype: int64
3117


,oid_,nri_id,state,stateabbrv,statefips,county,countytype,countyfips,stcofips,population,...,sovi_value,resl_npctl,cbsa_code,county_fips5,place_name,place_id,yr,index_nsa,index_prev,hpi_change
0,1,C01001,Alabama,AL,1,Autauga,County,1,01001,58764,...,NaN,NaN,33860,01001,"Montgomery, AL",33860,2023,205.86250,195.3500,10.51250
1,2,C01003,Alabama,AL,1,Baldwin,County,3,01003,231365,...,NaN,NaN,19300,01003,"Daphne-Fairhope-Foley, AL",19300,2023,363.14250,333.1125,30.03000
3,4,C01007,Alabama,AL,1,Bibb,County,7,01007,22239,...,NaN,NaN,13820,01007,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
4,5,C01009,Alabama,AL,1,Blount,County,9,01009,58992,...,NaN,NaN,13820,01009,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
5,8,C01015,Alabama,AL,1,Calhoun,County,15,01015,116250,...,NaN,NaN,11500,01015,"Anniston-Oxford, AL",11500,2023,261.24750,243.6425,17.60500
7,11,C01021,Alabama,AL,1,Chilton,County,21,01021,44999,...,NaN,NaN,13820,01021,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
9,17,C01033,Alabama,AL,1,Colbert,County,33,01033,57133,...,NaN,NaN,22520,01033,"Florence-Muscle Shoals, AL",22520,2023,271.35750,250.2950,21.06250
15,26,C01051,Alabama,AL,1,Elmore,County,51,01051,87755,...,NaN,NaN,33860,01051,"Montgomery, AL",33860,2023,205.86250,195.3500,10.51250
16,28,C01055,Alabama,AL,1,Etowah,County,55,01055,103320,...,NaN,NaN,23460,01055,"Gadsden, AL",23460,2023,290.03250,269.5775,20.45500
18,31,C01061,Alabama,AL,1,Geneva,County,61,01061,26621,...,NaN,NaN,20020,01061,"Dothan, AL",20020,2023,242.38500,221.0125,21.37250


In [167]:
nri.to_csv("../../02_processed_data/nri_hpi_data_full.csv")

In [168]:
nri[nri["nri_ver"] == 2022]

,oid_,nri_id,state,stateabbrv,statefips,county,countytype,countyfips,stcofips,population,...,sovi_value,resl_npctl,cbsa_code,county_fips5,place_name,place_id,yr,index_nsa,index_prev,hpi_change
